# Constitution Evidence — Kaggle GPU experiment

Enable **GPU** and **Internet** in Kaggle. Upload the private source dataset and add it as notebook input. Kaggle may unpack the ZIP; the loader accepts either its ZIP or extracted source directory and records a source revision/fingerprint. This notebook uses only the French corpus, measures whether the full text fits Qwen, and saves provisional outputs.


In [ ]:
from pathlib import Path
import os, subprocess, sys, json, hashlib, time, zipfile, re, shutil
ROOT = Path('/kaggle/working/moroccan-constitution-rag')
INPUT = Path('/kaggle/input')
# Kaggle may mount a ZIP or unpack its files into a dataset directory.
SOURCE_PATH = None  # Set an exact /kaggle/input/... path only if discovery is ambiguous.
paths = [Path(SOURCE_PATH)] if SOURCE_PATH else sorted(INPUT.rglob('*.zip'))
archives = []
for path in paths:
    if not path.is_file():
        continue
    try:
        with zipfile.ZipFile(path) as candidate:
            names = set(candidate.namelist())
            revision = candidate.comment.decode('ascii').strip()
            if {'pyproject.toml', 'notebooks/kaggle_gpu.ipynb'} <= names and re.fullmatch(r'[0-9a-f]{40}', revision):
                archives.append(path)
    except (OSError, zipfile.BadZipFile, UnicodeDecodeError):
        continue
folders = ([Path(SOURCE_PATH)] if SOURCE_PATH and Path(SOURCE_PATH).is_dir() else
           [path.parent for path in INPUT.rglob('pyproject.toml')
            if (path.parent/'notebooks/kaggle_gpu.ipynb').is_file()])
if len(archives) + len(folders) != 1:
    visible = [str(path) for path in sorted(INPUT.rglob('*')) if path.is_file()][:30]
    raise RuntimeError(f'Expected one source ZIP or extracted source directory; '
                       f'found {len(archives)} ZIPs and {len(folders)} directories. '
                       f'Visible input files: {visible}. Set SOURCE_PATH to the correct path if needed.')
if archives:
    with zipfile.ZipFile(archives[0]) as source_zip:
        SOURCE_REVISION = source_zip.comment.decode('ascii').strip()
        if any(Path(name).is_absolute() or '..' in Path(name).parts for name in source_zip.namelist()):
            raise RuntimeError('Source archive contains an unsafe path')
        ROOT.mkdir(parents=True, exist_ok=True)
        source_zip.extractall(ROOT)
    print('Source archive:', archives[0])
else:
    source_dir = folders[0]
    fingerprint = hashlib.sha256()
    for file in sorted(path for path in source_dir.rglob('*') if path.is_file()):
        relative = file.relative_to(source_dir).as_posix()
        fingerprint.update(relative.encode('utf-8') + b'\0')
        fingerprint.update(hashlib.sha256(file.read_bytes()).digest())
    SOURCE_REVISION = 'tree-sha256:' + fingerprint.hexdigest()
    shutil.copytree(source_dir, ROOT, dirs_exist_ok=True)
    print('Source directory:', source_dir)
os.chdir(ROOT)
print('Source revision:', SOURCE_REVISION)
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout.strip())


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.','huggingface_hub'],check=True,cwd=ROOT)
# Kaggle's live kernel may not refresh site-packages after a subprocess pip install.
sys.path.insert(0, str(ROOT/'src'))
import importlib
importlib.invalidate_caches()
import constitution_evidence, bm25s
print('Project import:', constitution_evidence.__file__)
subprocess.run(['apt-get','update','-qq'],check=True,stdout=subprocess.DEVNULL)
subprocess.run(['apt-get','install','-y','-qq','poppler-utils','tesseract-ocr','tesseract-ocr-fra'],check=True,stdout=subprocess.DEVNULL)
(ROOT/'data/raw').mkdir(parents=True,exist_ok=True)
pdf = ROOT/'data/raw/constitution_2011_fr_marocma.pdf'
if not pdf.exists():
    subprocess.run(['curl','-L','--fail','-o',str(pdf),'https://maroc.ma/sites/default/files/2024-10/constitution_du_1er_juillet_2011.pdf'],check=True)
assert hashlib.sha256(pdf.read_bytes()).hexdigest() == '4101d45b4733a6b36eee77bd9ef4175b23160103089b003b13907e8b6d59333d'
# Tesseract's installed French pack is used instead of downloading a second copy.
from constitution_evidence.ingestion import ingest
from constitution_evidence.store import read_jsonl
import shutil
language_dir = Path('/usr/share/tesseract-ocr/5/tessdata')
manifest = ingest(ROOT/'configs/corpus.json',root=ROOT,tessdata=language_dir)
print(manifest['article_count'],manifest['passage_count'],manifest['corpus_hash'])


In [ ]:
from huggingface_hub import HfApi, hf_hub_download
repo = 'Qwen/Qwen3-4B-GGUF'
revision = HfApi().model_info(repo).sha
model_path = hf_hub_download(repo_id=repo,filename='Qwen3-4B-Q4_K_M.gguf',revision=revision)
digest = hashlib.sha256()
with open(model_path, 'rb') as stream:
    for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
        digest.update(block)
model_hash = digest.hexdigest()
print(revision,model_hash)


In [ ]:
source = Path('/kaggle/working/llama.cpp')
if not source.exists(): subprocess.run(['git','clone','--depth','1','https://github.com/ggml-org/llama.cpp.git',str(source)],check=True)
if not shutil.which('nvcc'): raise RuntimeError('CUDA compiler unavailable in this Kaggle image; select a GPU image with CUDA toolkit')
build = Path('/kaggle/working/llama-build')
# Kaggle's CUDA toolkit may lack the libcuda development target; disable VMM's direct driver link.
subprocess.run(['cmake','-S',str(source),'-B',str(build),'-DGGML_CUDA=ON','-DGGML_CUDA_NO_VMM=ON','-DCMAKE_BUILD_TYPE=Release'],check=True)
subprocess.run(['cmake','--build',str(build),'-j','4','--target','llama-server'],check=True)
server = subprocess.Popen([str(build/'bin/llama-server'),'-m',model_path,'--alias','Qwen3-4B','--host','127.0.0.1','--port','8080','-c','32768','-ngl','99','--jinja'],stdout=open('/kaggle/working/llama-server.log','w'),stderr=subprocess.STDOUT)
import urllib.request
for _ in range(180):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8080/health',timeout=1) as response:
            if response.status == 200: break
    except Exception: pass
    time.sleep(2)
else: raise RuntimeError('llama-server did not become healthy; inspect /kaggle/working/llama-server.log')
print('llama-server ready')


In [ ]:
from constitution_evidence.retrieval import BM25
from constitution_evidence.evidence import pack
from constitution_evidence.local_generation import LocalGenerator, LOCAL_PROMPT_VERSION
from constitution_evidence.store import read_jsonl, write_jsonl, digest
passages = read_jsonl(ROOT/'data/processed/passages.jsonl')
questions = read_jsonl(ROOT/'data/annotations/calibration.jsonl')
engine = BM25(passages)
generator = LocalGenerator(model='Qwen3-4B',timeout=300)
run_dir = ROOT/'runs/kaggle-qwen4b-calibration'
run_dir.mkdir(parents=True,exist_ok=True)
run_manifest = {
    'model_repo':repo, 'model_revision':revision, 'model_sha256':model_hash,
    'corpus_hash':manifest['corpus_hash'], 'passages_hash':manifest['passages_hash'],
    'questions_sha256':digest((ROOT/'data/annotations/calibration.jsonl').read_bytes()),
    'source_revision':SOURCE_REVISION,
    'llama_commit':subprocess.run(['git','-C',str(source),'rev-parse','HEAD'],capture_output=True,text=True).stdout.strip(),
    'review_status':manifest['review_status'], 'question_count':len(questions),
    'strategy':'bm25', 'top_k':5, 'context_chars':7000, 'prompt_version':LOCAL_PROMPT_VERSION,
    'status':'provisional_generation_calibration'
}
manifest_path = run_dir/'manifest.json'
if manifest_path.exists():
    assert json.loads(manifest_path.read_text()) == run_manifest, 'Run directory belongs to different inputs'
else:
    manifest_path.write_text(json.dumps(run_manifest,indent=2)+'\n')
rows_path = run_dir/'responses.jsonl'
rows = read_jsonl(rows_path) if rows_path.exists() else []
done = {row['question_id'] for row in rows}
assert len(done) == len(rows), 'Duplicate question IDs in saved run'
for q in questions:
    if q['question_id'] in done: continue
    hits = engine.search(q['question'],5)
    context = pack(hits,7000)
    row = {'question_id':q['question_id'], 'context':[
        {'passage_id':p['passage_id'],'text_hash':p['text_hash'],'canonical_text':p['canonical_text']}
        for p in context]}
    started = time.monotonic()
    try:
        row['answer'] = generator.generate(q['question'],context,manifest)
    except Exception as exc:
        row['error'] = {'type':type(exc).__name__,'message':str(exc)}
    row['elapsed_seconds'] = round(time.monotonic()-started,3)
    rows.append(row)
    write_jsonl(rows_path,rows)
print('Saved',len(rows),'of',len(questions),'provisional responses to',rows_path)


Review `runs/kaggle-qwen4b-calibration/` and the source PDF manually. The calibration run resumes after interruption and keeps exact supplied context and model hashes. The full-text smoke file records token capacity and one raw answer or the failure. These 15 calibration questions are provisional; the notebook does not score legal faithfulness or replace the reviewed French benchmark.


In [ ]:
# One full-text capacity and answer smoke check; this does not open the reserved test.
from constitution_evidence.local_generation import LocalGenerator
from constitution_evidence.generation import GenerationError
all_context = "\n\n".join(f"[{p['passage_id']}]\n{p['canonical_text']}" for p in passages)
fulltext_generator = LocalGenerator(model='Qwen3-4B', max_context_chars=len(all_context)+1,
                                    timeout=600, require_token_preflight=True)
smoke_question = questions[0]['question']
capacity_path = ROOT/'runs/kaggle-qwen4b-calibration/fulltext_smoke.json'
capacity_result = {'question_id':questions[0]['question_id'], 'model_sha256':model_hash,
                   'corpus_hash':manifest['corpus_hash'], 'passages_hash':manifest['passages_hash'],
                   'passage_count':len(passages)}
try:
    capacity_result['preflight'] = fulltext_generator.preflight(
        fulltext_generator.messages(smoke_question, passages))
    capacity_result['answer'] = fulltext_generator.generate(smoke_question, passages, manifest)
except Exception as exc:
    capacity_result['error'] = {'type':type(exc).__name__, 'message':str(exc)}
capacity_path.write_text(json.dumps(capacity_result,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({key:value for key,value in capacity_result.items() if key != 'answer'},indent=2))
print('Saved',capacity_path)
